# Sample submission file for 2026 machine learning project
## Introduction to Machine Learning and Data Science, UMONS, 2026

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

We first load the training and test datasets.

In [2]:
train_df = pd.read_csv("results_train.csv")
test_df = pd.read_csv("results_test.csv")

# Prepare 'Id' column for both train and test sets for merging
train_df['Id'] = train_df['Gemeinde-Nummer'].astype(str)
test_df['Id'] = test_df['Gemeinde-Nummer'].astype(str)

print(f"Loaded train set: {train_df.shape}")
print(f"Loaded test set: {test_df.shape}")

Loaded train set: (1559, 14)
Loaded test set: (669, 6)


In [3]:
train_df.head()

,Kantons-Nummer,Kanton,Gemeinde-Nummer,Gemeinde,Stimmberechtigte,eingelegte Stimmzettel,Stimmbeteiligung,leere Stimmzettel,ungültige Stimmzettel,gültige Stimmen,Ja-Stimmen,Nein-Stimmen,Ja in Prozent,Id
0,3,Luzern,1142,Roggliswil,525.0,265.0,50.476190,1.0,4.0,260.0,69.0,191.0,26.538462,1142
1,10,Fribourg / Freiburg,2261,Greng,131.0,79.0,60.305344,0.0,0.0,79.0,28.0,51.0,35.443038,2261
2,23,Valais / Wallis,6300,Zermatt,2910.0,1242.0,42.680412,24.0,29.0,1189.0,639.0,550.0,53.742641,6300
3,11,Solothurn,2556,Selzach,2427.0,999.0,41.161928,24.0,1.0,974.0,446.0,528.0,45.790554,2556
4,17,St. Gallen,3217,Steinach,2261.0,950.0,42.016807,23.0,0.0,927.0,356.0,571.0,38.403452,3217


In [4]:
test_df.head()

,Kantons-Nummer,Kanton,Gemeinde-Nummer,Gemeinde,Stimmberechtigte,Id
0,1,Zürich,69,Wallisellen,9182.0,69
1,2,Bern / Berne,981,Niederbipp,3028.0,981
2,22,Vaud,5432,Montherod,269.0,5432
3,2,Bern / Berne,387,Lengnau (BE),3335.0,387
4,10,Fribourg / Freiburg,2194,Ferpicloz,186.0,2194


We load two of the four datasets (other referendum, data about the communes). Of course, you should use all datasets; this is simply a minimal example to show how to join two datasets together.

In [5]:
# 1. Load the other referendum
file_622 = "622.00-result-by-canton-district-and-municipality.xlsx"
df_622 = pd.read_excel(file_622, sheet_name="Gemeinden", header=5)
df_622.columns = df_622.columns.str.strip()

# Clean IDs (this could be factored into a function since we need to do it for multiple datasets)
df_622['Gemeinde-Nummer'] = pd.to_numeric(df_622['Gemeinde-Nummer'], errors='coerce')
df_622 = df_622.dropna(subset=['Gemeinde-Nummer'])
df_622['Id'] = df_622['Gemeinde-Nummer'].astype(int).astype(str)

# Suffix columns to distinguish them from the target dataset and drop duplicate texts
df_622 = df_622.add_suffix('_622')
df_622 = df_622.rename(columns={'Id_622': 'Id'})
df_622 = df_622.drop(columns=['Gemeinde-Nummer_622', 'Gemeinde_622', 'Kanton_622'])


# 2. Load "Portraits of the communes" dataset
file_jee = "je-e-21.03.01.xlsx"
df_jee = pd.read_excel(file_jee, sheet_name="Schweiz - Gemeinden", header=5)

# Clean IDs
df_jee['Number of commune'] = pd.to_numeric(df_jee['Number of commune'], errors='coerce')
df_jee = df_jee.dropna(subset=['Number of commune'])
df_jee['Id'] = df_jee['Number of commune'].astype(int).astype(str)
df_jee = df_jee.drop(columns=['Number of commune', 'Name of commune'])

# Force columns to numeric (replaces some '*' with NaN)
for col in df_jee.columns:
    if col != 'Id':
        df_jee[col] = pd.to_numeric(df_jee[col], errors='coerce')

In [6]:
print(df_622.shape)
df_622.head()

(2228, 11)


,Kantons-Nummer_622,Stimmberechtigte_622,eingelegte Stimmzettel_622,Stimmbeteiligung_622,leere Stimmzettel_622,ungültige Stimmzettel_622,gültige Stimmen_622,Ja-Stimmen_622,Nein-Stimmen_622,Ja in Prozent_622,Id
1,1,1402.0,662.0,47.218260,5.0,0.0,657.0,206.0,451.0,31.354642,1
2,1,7186.0,2877.0,40.036181,49.0,0.0,2828.0,695.0,2133.0,24.575672,2
3,1,3656.0,1690.0,46.225383,25.0,1.0,1664.0,404.0,1260.0,24.278846,3
4,1,2505.0,1140.0,45.508982,14.0,0.0,1126.0,353.0,773.0,31.349911,4
5,1,2515.0,1180.0,46.918489,12.0,0.0,1168.0,312.0,856.0,26.712329,5


In [7]:
print(df_jee.shape)
df_jee.head()

(2240, 42)


,Residents,Change in %,Population density per km²,Foreign nationals in %,0-19 years,20-64 years,65 years or over,Crude marriage rate,Crude divorce rate,Crude birth rate,...,CVP,SP,SVP,EVP/CSP,GLP,BDP,PdA/Sol.,GPS,Small right-wing parties,Id
3,1977.0,8.388158,249.936789,13.100658,20.586748,62.822458,16.590794,2.526529,3.031834,7.074280,...,2.076428,18.645940,30.929249,3.467063,8.435250,2.617442,0.167638,7.075094,4.888178,1
4,11900.0,7.294203,1123.701605,27.848740,20.285714,62.201681,17.512605,5.167740,1.440190,11.860386,...,4.585387,19.080314,33.785785,5.464827,7.357860,4.164299,0.190049,6.211047,1.768197,2
5,5435.0,5.349874,731.493943,14.149034,23.808648,60.717571,15.473781,5.389834,1.858563,10.779667,...,3.378541,20.403265,29.100156,3.143003,11.862398,3.803108,0.112518,6.661066,1.915807,3
6,3571.0,6.279762,262.573529,14.533744,22.738729,60.403248,16.858023,4.540295,1.986379,7.661748,...,2.881915,19.393305,34.937369,2.569875,8.748273,4.656087,0.193911,8.021665,1.825436,4
7,3687.0,8.123167,564.624809,14.971522,22.484405,62.110117,15.405479,7.622159,1.361100,7.349939,...,3.918166,22.478008,30.114599,3.589299,9.625938,3.768864,0.227988,6.466387,1.840045,5


We merge the datasets on the common key `Id` with a left join, and we clean up the columns not available in the test set (i.e., the voting results). We then select only the columns with numeric data, and we impute missing values with the mean of each column.

In [8]:
# Merge datasets
train_merged = train_df.merge(df_622, on='Id', how='left').merge(df_jee, on='Id', how='left')
test_merged = test_df.merge(df_622, on='Id', how='left').merge(df_jee, on='Id', how='left')

# Define target variable
y_train = train_merged['Ja in Prozent']

leakage_columns = [
    'eingelegte Stimmzettel', 'Stimmbeteiligung', 'leere Stimmzettel', 
    'ungültige Stimmzettel', 'gültige Stimmen', 'Ja-Stimmen', 'Nein-Stimmen', 'Ja in Prozent'
]

# Select only number columns and remove voting results
X_train_raw = train_merged.select_dtypes(include=[np.number]).drop(columns=[c for c in leakage_columns if c in train_merged.columns])

# Ensure test set has exactly the same feature columns
X_test_raw = test_merged[X_train_raw.columns]

# Impute missing values (replace NaNs with mean of the column)
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train_raw)
X_test = imputer.transform(X_test_raw)

print(f"Training shape: {X_train.shape}, test shape: {X_test.shape}")

Training shape: (1559, 53), test shape: (669, 53)


For the predictions: very simple model that always predicts the mean of the target variable in the training set. This is a common baseline model to compare against more complex models. Note that, here, we do not really use `X_train`, but you should of course use it to train a more complex model.

In [9]:
mean_prediction = y_train.mean()
print(f"Baseline mean prediction: {mean_prediction:.4f}%")

test_predictions = np.full(shape=X_test.shape[0], fill_value=mean_prediction)

# Format for Kaggle submission
submission_df = pd.DataFrame({
    'Id': test_merged['Id'],
    'Predicted': test_predictions
})

submission_df.to_csv("submission_mean_baseline.csv", index=False)
print("Saved submission_mean_baseline.csv for Kaggle.")

display(submission_df.head())

Baseline mean prediction: 40.9146%
Saved submission_mean_baseline.csv for Kaggle.


,Id,Predicted
0,69,40.914616
1,981,40.914616
2,5432,40.914616
3,387,40.914616
4,2194,40.914616
